In [9]:
from pathlib import Path

import pandas as pd
import numpy as np

# League standings
The purpose of this notebook is to use our datasets to extract the final season's standings and see if it is consistent with real life results. This helps us check if our data is valid.

In [8]:
raw_data_folder = Path.cwd().parent/"data"/"raw"/"ligue1"

raw_l1_data = {}
for file in raw_data_folder.glob("*.csv"):
    raw_l1_data[file.stem] = pd.read_csv(file)
    print(f"Extracted {file.stem} with shape {raw_l1_data[file.stem].shape}")

Extracted F1_2122 with shape (380, 105)
Extracted F1_2223 with shape (380, 105)
Extracted F1_2324 with shape (306, 105)
Extracted F1_2425 with shape (306, 119)
Extracted F1_2526 with shape (305, 131)


# Number of teams check
## Recovering the Number of Teams from the Number of Matches

For a double round-robin league with \(n\) teams, each team plays every other team twice (home and away).

The total number of matches is therefore:

$$
n(n-1)=k
$$

where \(k\) is the number of matches in the dataset.

Rearranging:

$$
n^2 - n - k = 0
$$

Using the quadratic formula:

$$
n = \frac{1 \pm \sqrt{1+4k}}{2}
$$

Since the number of teams must be a strictly positive integer, we keep only the positive solution:

$$
n = \frac{1 + \sqrt{1+4k}}{2}
$$

For example, if the dataset contains \(380\) matches:

$$
n = \frac{1 + \sqrt{1+4 \times 380}}{2}
= \frac{1 + \sqrt{1521}}{2}
= \frac{1 + 39}{2}
= 20
$$

Therefore, the league contains \(20\) teams. In the test below, we are going to check if the number we get from the formula is the same as the unique values of home and away teams in the data

In [21]:
def get_nb_teams(match_count: int) -> int:
    n = (1+np.sqrt(1+4*match_count))/2
    if not np.isclose(n, round(n)):
        print(
            f"{match_count} matches does not correspond to a complete double round-robin season."
        )

    return int(round(n))

In [22]:
get_nb_teams(raw_l1_data['F1_2122'].shape[0])

20

In [23]:
for name, df in raw_l1_data.items():
    n = get_nb_teams(df.shape[0])
    print(f"{name} has k={df.shape[0]} rows, therefore we get {n} teams.")
    unique_teams = len(set(df['HomeTeam']) | set(df['AwayTeam']))
    print(f"{name} has {unique_teams} unique teams.")
    if unique_teams == n:
        print("Test Worked")
    else:
        print("Problem")

F1_2122 has k=380 rows, therefore we get 20 teams.
F1_2122 has 20 unique teams.
Test Worked
F1_2223 has k=380 rows, therefore we get 20 teams.
F1_2223 has 20 unique teams.
Test Worked
F1_2324 has k=306 rows, therefore we get 18 teams.
F1_2324 has 18 unique teams.
Test Worked
F1_2425 has k=306 rows, therefore we get 18 teams.
F1_2425 has 18 unique teams.
Test Worked
305 matches does not correspond to a complete double round-robin season.
F1_2526 has k=305 rows, therefore we get 18 teams.
F1_2526 has 18 unique teams.
Test Worked


We see that we have one missing match potentially from the last dataset

In [20]:
raw_l1_data['F1_2526']['AwayTeam'].value_counts().add(
    raw_l1_data['F1_2526']['HomeTeam'].value_counts()
).sort_values()

AwayTeam
Nantes        33
Toulouse      33
Brest         34
Angers        34
Lens          34
Lille         34
Lorient       34
Auxerre       34
Lyon          34
Marseille     34
Metz          34
Monaco        34
Nice          34
Paris FC      34
Paris SG      34
Le Havre      34
Rennes        34
Strasbourg    34
Name: count, dtype: int64

### Toulouse VS Nantes 25/26 cancelled
We can see from the data that this match was cancelled since we are missing this match. The link for info https://ligue1.com/fr/articles/l1_article_5119-j34-nantes-toulouse-definitivement-arrete?.